<a href="https://colab.research.google.com/github/chilukuru-saikrishna/regulatory-document-qa/blob/main/Regulatory_Document_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📜 Regulatory Document QA
A RAG (retrieval-augmented generation) agent that answers questions over long iGaming regulatory PDFs and cites the document and page for every claim.

**Stack:** Python · PyMuPDF · ChromaDB · Gemini API · Gradio — runs entirely in Google Colab.

**Documents:** UKGC Licence Conditions and Codes of Practice · MGA Player Protection Directive · FIAU Implementing Procedures Part II (Remote Gaming)

**Before you start:** add your Gemini API key in Colab's 🔑 **Secrets** panel with the name `GEMINI_API_KEY` and turn *Notebook access* on.

**Each new session:** run cells 1, 2, 4, 6, 7, 8. Skip 3 and 5 (your PDFs and index are already saved in Drive).

## Cell 1 — Install libraries

In [ ]:
!pip install -q pymupdf chromadb google-genai gradio

## Cell 2 — Settings, Google Drive and API key

In [ ]:
import os, hashlib
from pathlib import Path
from google.colab import userdata, drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/regulatory-document-qa')
DATA_DIR = BASE / 'data'
DB_PATH = str(BASE / 'chroma_db')
DATA_DIR.mkdir(parents=True, exist_ok=True)

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
GEN_MODEL = 'gemini-flash-latest'      # model that writes answers
EMBED_MODEL = 'gemini-embedding-001'   # model that turns text into numbers
print('Working folder:', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/regulatory-document-qa


## Cell 3 — Upload PDFs (first time only)
You can also drop PDFs straight into `MyDrive/regulatory-document-qa/data/` in Google Drive and skip this cell.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name, content in uploaded.items():
    (DATA_DIR / name).write_bytes(content)
print(sorted(p.name for p in DATA_DIR.glob('*.pdf')))

Saving UKGC_LCCP.pdf to UKGC_LCCP (1).pdf
['MGA_Player_Protection_Directive.pdf', 'UKGC_LCCP (1).pdf', 'UKGC_LCCP.pdf']


## Cell 4 — Building blocks: extract, chunk, embed, store

In [ ]:
import fitz  # PyMuPDF
import time
import chromadb
from google import genai
from google.genai import types
from google.genai.errors import ClientError

client = genai.Client()
db = chromadb.PersistentClient(path=DB_PATH)
collection = db.get_or_create_collection('regulatory_docs', metadata={'hnsw:space': 'cosine'})

CHUNK_SIZE, OVERLAP = 1200, 200
BATCH = 20   # chunks per request
PAUSE = 15   # 20 chunks every 15s = 80 per minute, under the 100/min free limit

def extract_pages(pdf_path):
    """Yield (page_number, text); 1-based so citations match the PDF viewer."""
    with fitz.open(pdf_path) as doc:
        for i, page in enumerate(doc, start=1):
            text = ' '.join(page.get_text('text').split())
            if text:
                yield i, text

def chunk_text(text, size=CHUNK_SIZE, overlap=OVERLAP):
    """Split text into overlapping pieces."""
    for start in range(0, len(text), size - overlap):
        yield text[start:start + size]

def embed(texts, task_type, max_retries=6):
    """Embed texts with Gemini; on a rate limit, wait and retry just this batch."""
    wait = 60
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.models.embed_content(
                model=EMBED_MODEL, contents=texts,
                config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=768))
            return [e.values for e in resp.embeddings]
        except ClientError as e:
            if getattr(e, 'code', None) == 429 or 'RESOURCE_EXHAUSTED' in str(e):
                print(f'   ⏳ Rate limit — waiting {wait}s (try {attempt}/{max_retries})')
                time.sleep(wait)
                wait = min(wait * 2, 120)
            else:
                raise
    raise RuntimeError('Still rate-limited after several retries. '
                       'You may have used up the daily free quota — try again tomorrow.')

def ingest_pdf(pdf_path):
    """Extract, chunk, embed and store one PDF, skipping chunks already saved."""
    ids, docs, metas = [], [], []
    for page_no, text in extract_pages(pdf_path):
        for j, chunk in enumerate(chunk_text(text)):
            ids.append(hashlib.md5(f'{pdf_path.name}-{page_no}-{j}'.encode()).hexdigest())
            docs.append(chunk)
            metas.append({'source': pdf_path.name, 'page': page_no})

    existing = set(collection.get(ids=ids, include=[])['ids'])
    todo = [i for i, uid in enumerate(ids) if uid not in existing]
    print(f'{pdf_path.name}: {len(ids)} chunks total, {len(existing)} already saved, {len(todo)} to embed')

    for start in range(0, len(todo), BATCH):
        idx = todo[start:start + BATCH]
        batch_ids = [ids[i] for i in idx]
        batch_docs = [docs[i] for i in idx]
        batch_metas = [metas[i] for i in idx]
        batch_vecs = embed(batch_docs, 'RETRIEVAL_DOCUMENT')
        collection.upsert(ids=batch_ids, documents=batch_docs,
                          embeddings=batch_vecs, metadatas=batch_metas)
        print(f'   saved {min(start + BATCH, len(todo))}/{len(todo)}')
        time.sleep(PAUSE)

print('Chunks already in database:', collection.count())

Chunks already in database: 224


## Cell 5 — Index the PDFs (first time, or after adding new PDFs)

In [ ]:
for pdf in sorted(DATA_DIR.glob('*.pdf')):
    ingest_pdf(pdf)
print('Total chunks:', collection.count())

MGA_Player_Protection_Directive.pdf: 74 chunks total, 74 already saved, 0 to embed
UKGC_LCCP (1).pdf: 244 chunks total, 244 already saved, 0 to embed
UKGC_LCCP.pdf: 244 chunks total, 244 already saved, 0 to embed
Total chunks: 562


## Cell 6 — Retrieve and answer with citations

In [ ]:
import time
from google.genai.errors import ServerError, ClientError

PROMPT = """You are a regulatory compliance assistant.
Answer the question using ONLY the context below.
Cite every claim inline as [source, p.X] using the
source name and page given for each excerpt.
If the context does not contain the answer, say
"Not found in the provided documents." Do not guess.

Context:
{context}

Question: {question}
Answer:"""

SKIP_WORDS = ['image', 'tts', 'audio', 'live',
              'embedding', 'vision', 'learnlm']


def find_flash_models():
    """List text 'flash' models this API key can use."""
    found = []
    for m in client.models.list():
        actions = m.supported_actions or []
        if 'generateContent' not in actions:
            continue
        name = m.name.replace('models/', '')
        if 'flash' not in name:
            continue
        if any(w in name for w in SKIP_WORDS):
            continue
        found.append(name)
    # stable models first, previews last
    found.sort(key=lambda n: ('preview' in n, n))
    return found


GEN_MODELS = ['gemini-3.6-flash', GEN_MODEL, 'gemini-flash-lite-latest']
for name in find_flash_models():
    if name not in GEN_MODELS:
        GEN_MODELS.append(name)

print('Models to try, in order:')
for name in GEN_MODELS:
    print('  -', name)


def retrieve(question, k=6):
    """Find the k chunks most similar to the question."""
    q_vec = embed([question], 'RETRIEVAL_QUERY')[0]
    res = collection.query(
        query_embeddings=[q_vec],
        n_results=k,
        include=['documents', 'metadatas', 'distances'],
    )
    docs = res['documents'][0]
    metas = res['metadatas'][0]
    dists = res['distances'][0]

    hits = []
    for d, m, dist in zip(docs, metas, dists):
        hits.append({
            'text': d,
            'source': m['source'],
            'page': m['page'],
            'score': 1 - dist,
        })
    return hits


def generate(prompt, retries_per_model=2):
    """Ask Gemini; retry when busy, then try next model."""
    for model in GEN_MODELS:
        wait = 5
        for attempt in range(1, retries_per_model + 1):
            try:
                resp = client.models.generate_content(
                    model=model,
                    contents=prompt,
                )
                print(f'  Answered by: {model}')
                return resp.text
            except (ServerError, ClientError) as e:
                code = getattr(e, 'code', None)
                if code in (429, 500, 503):
                    print(f'  {model} busy ({code}), '
                          f'retry in {wait}s')
                    time.sleep(wait)
                    wait = wait * 2
                elif code == 404:
                    print(f'  {model} not available, skipping')
                    break
                else:
                    raise
        print(f'  Moving on from {model}')
    raise RuntimeError('All models busy. Try again later.')


def answer(question, k=6):
    """Retrieve context and get a cited answer."""
    hits = retrieve(question, k)
    parts = []
    for h in hits:
        label = f"[{h['source']}, p.{h['page']}]"
        parts.append(label + '\n' + h['text'])
    context = '\n\n'.join(parts)
    prompt = PROMPT.format(context=context,
                           question=question)
    text = generate(prompt)
    return text, hits

Models to try, in order:
  - gemini-3.6-flash
  - gemini-flash-latest
  - gemini-flash-lite-latest
  - gemini-2.5-flash
  - gemini-2.5-flash-lite
  - gemini-3.1-flash-lite
  - gemini-3.5-flash
  - gemini-3.5-flash-lite
  - gemini-3.7-flash
  - gemini-3.8-flash
  - gemini-omni-1.1-flash
  - gemini-3-flash-preview
  - gemini-3.1-flash-lite-preview
  - gemini-omni-flash-preview


## Cell 7 — Ask a question in the notebook

In [ ]:
from IPython.display import Markdown, display

def ask(question, k=6):
    text, hits = answer(question, k)
    display(Markdown(f'**Q:** {question}\n\n{text}'))
    rows = '\n'.join(f"| {h['source']} | {h['page']} | {h['score']:.2f} |" for h in hits)
    display(Markdown('| Source | Page | Similarity |\n|---|---|---|\n' + rows))

ask('What are the markers of harm licensees must consider?')

  Answered by: gemini-3.6-flash


**Q:** What are the markers of harm licensees must consider?

Licensees must use a range of indicators to identify harm or potential harm associated with gambling, which must include the following:

* Customer spend [UKGC_LCCP (1).pdf, p.57] [UKGC_LCCP.pdf, p.57]
* Patterns of spend [UKGC_LCCP (1).pdf, p.57] [UKGC_LCCP.pdf, p.57]
* Time spent gambling [UKGC_LCCP (1).pdf, p.57] [UKGC_LCCP.pdf, p.57]
* Gambling behaviour indicators [UKGC_LCCP (1).pdf, p.57] [UKGC_LCCP.pdf, p.57]
* Customer-led contact [UKGC_LCCP (1).pdf, p.57] [UKGC_LCCP.pdf, p.57]
* Use of gambling management tools [UKGC_LCCP (1).pdf, p.57] [UKGC_LCCP.pdf, p.57]
* Account indicators [UKGC_LCCP (1).pdf, p.57] [UKGC_LCCP.pdf, p.57]

Additionally, licensees must consider factors that might make a customer more vulnerable to experiencing gambling harms and act when indicators of vulnerability are identified [UKGC_LCCP (1).pdf, p.57] [UKGC_LCCP.pdf, p.57].

| Source | Page | Similarity |
|---|---|---|
| UKGC_LCCP (1).pdf | 58 | 0.75 |
| UKGC_LCCP.pdf | 58 | 0.75 |
| UKGC_LCCP (1).pdf | 57 | 0.74 |
| UKGC_LCCP.pdf | 57 | 0.74 |
| UKGC_LCCP (1).pdf | 57 | 0.73 |
| UKGC_LCCP.pdf | 57 | 0.73 |

## Cell 8 — Gradio web app
Runs inside the notebook and prints a public `gradio.live` link (valid 72 hours, while this session is running).

In [ ]:
import gradio as gr

def qa_app(question, k):
    if not question.strip():
        return "Please type a question.", None
    text, hits = answer(question, int(k))
    sources = [[h['source'], h['page'], round(h['score'], 2)] for h in hits]
    return text, sources

with gr.Blocks(title="Regulatory Document QA") as demo:
    gr.Markdown("# 📜 Regulatory Document QA\n"
                "Ask questions about UKGC, MGA and FIAU regulatory documents. "
                "Every answer cites the document and page.")
    with gr.Row():
        question = gr.Textbox(label="Your question", placeholder="e.g. What are the markers of harm?", scale=4)
        k = gr.Slider(3, 12, value=6, step=1, label="Chunks to retrieve", scale=1)
    ask_btn = gr.Button("Ask", variant="primary")
    answer_box = gr.Markdown()
    sources_table = gr.Dataframe(headers=["Source", "Page", "Similarity"], label="Retrieved sources")

    gr.Examples(
        examples=["What are the markers of harm licensees must consider?",
                  "When must a licensee report a suspicion of money laundering?",
                  "What must be shown to a player before their first wager?"],
        inputs=question)

    ask_btn.click(qa_app, inputs=[question, k], outputs=[answer_box, sources_table])
    question.submit(qa_app, inputs=[question, k], outputs=[answer_box, sources_table])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e91d93641ea28680f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  gemini-3.6-flash busy (503), retry in 5s
  gemini-3.6-flash busy (503), retry in 10s
  Moving on from gemini-3.6-flash
  Answered by: gemini-flash-latest
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://e91d93641ea28680f3.gradio.live
